# Notebook 09 — ODE Models of Biological Systems

**Module:** 12 — Systems and Network Biology  
**Tier:** 2 — Working competence  
**Notebook:** 09 of 12  
**Time estimate:** 90 minutes

> Boolean models are useful but coarse. Ordinary differential equations let us model
> continuous concentrations, quantitative regulation, and the rich dynamical behavior
> that emerges from nonlinear biochemical kinetics.

---
## Step 1 — Motivation

Real gene expression levels are continuous, not binary. Chemical concentrations vary
quantitatively. ODE models describe these continuous dynamics and make quantitative,
testable predictions. Three canonical examples — Lotka-Volterra (ecology),
SIR (epidemiology), and the repressilator (synthetic biology) — illustrate the
richness of even simple nonlinear ODE systems.

---
## Step 2 — Intuition

An ODE system describes how rates of change depend on current state:
$$\frac{d\mathbf{x}}{dt} = \mathbf{f}(\mathbf{x}, \mathbf{p})$$

The key question: does the system go to a fixed point (steady state), oscillate
(limit cycle), or show chaotic behavior? These are determined by the topology of
the feedback network and the nonlinearity of interactions.

- **Fixed point:** $\mathbf{f}(\mathbf{x}^*) = 0$ — equilibrium
- **Limit cycle:** periodic trajectory in state space — oscillation
- **Bifurcation:** a parameter change that qualitatively changes the dynamics
  (e.g., from stable fixed point to oscillation)

---
## Step 3 — Biological Background

**Lotka-Volterra (prey-predator):**
- Predator-prey population dynamics; models parasite-host, immune cell-pathogen
- Conserved quantity: oscillations are neutral (neither stable nor unstable)

**SIR model:**
- Susceptible → Infected → Recovered — classic epidemiological model
- Basic reproduction number $R_0 = \beta N / \gamma$ determines whether epidemic spreads
- Extended models: SEIR (exposed), SIRS (waning immunity), network-SIR (heterogeneous contacts)

**Repressilator (Elowitz & Leibler 2000):**
- First synthetic genetic oscillator — constructed in E. coli
- Three genes (lacI, tetR, cI) mutually repress each other in a ring
- Sustained oscillations with ~150-minute period
- Proof of principle that gene networks can implement oscillators from design principles

**Michaelis-Menten kinetics:**
$$v = \frac{V_{max}[S]}{K_m + [S]}$$
- Saturation kinetics: rate increases sub-linearly with substrate concentration
- At $[S] \ll K_m$: linear (first-order)
- At $[S] \gg K_m$: saturated (zero-order), rate ≈ $V_{max}$

**Hill function (cooperative regulation):**
$$f(x) = \frac{x^n}{K^n + x^n}$$
- $n=1$: Michaelis-Menten; $n>1$: cooperative (sigmoidal, switch-like)
- Used in GRN models: $n=2$–$4$ for TF binding

---
## Step 4 — Mathematical Explanation

**Lotka-Volterra:**
$$\frac{dN}{dt} = \alpha N - \beta N P, \quad \frac{dP}{dt} = \delta N P - \gamma P$$

Fixed points: $(0,0)$ (trivial) and $(\gamma/\delta, \alpha/\beta)$ (coexistence).
The coexistence point is a **center** — neutral oscillations, conserved quantity
$H = \delta N - \gamma \ln N + \beta P - \alpha \ln P = \text{const}$.

**SIR:**
$$\frac{dS}{dt} = -\beta S I, \quad \frac{dI}{dt} = \beta S I - \gamma I, \quad \frac{dR}{dt} = \gamma I$$

Threshold: epidemic grows iff $\beta S / \gamma > 1$ (i.e., $S > \gamma/\beta$).
No exact analytic solution; integrate numerically.

**Repressilator (3-gene):**
$$\dot{m}_i = -m_i + \frac{\alpha}{1 + p_j^n} + \alpha_0, \quad \dot{p}_i = -\beta(p_i - m_i)$$

where $j = i-1 \mod 3$ (lacI represses tetR, tetR represses cI, cI represses lacI).
Oscillation requires sufficient cooperativity ($n > 1$) and appropriate $\alpha/\beta$ ratio.

**Linear stability analysis:** At fixed point $\mathbf{x}^*$, compute the Jacobian
$J_{ij} = \partial f_i / \partial x_j \big|_{\mathbf{x}^*}$.
If all eigenvalues have negative real part → stable fixed point.
If eigenvalues are purely imaginary → Hopf bifurcation (onset of oscillations).

In [ ]:
# Step 6 — Python: Three ODE models
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# ---- 1. Lotka-Volterra ----
def lotka_volterra(t, y, alpha, beta, delta, gamma):
    N, P = y  # prey, predator
    dN = alpha * N - beta * N * P
    dP = delta * N * P - gamma * P
    return [dN, dP]

lv_params = dict(alpha=1.0, beta=0.1, delta=0.075, gamma=1.5)
t_span = (0, 30)
t_eval = np.linspace(0, 30, 1000)
lv_sol = solve_ivp(lotka_volterra, t_span, [10, 5], t_eval=t_eval, args=tuple(lv_params.values()))
print(f'Lotka-Volterra: prey range [{lv_sol.y[0].min():.2f}, {lv_sol.y[0].max():.2f}]')

# ---- 2. SIR model ----
def sir_model(t, y, beta, gamma, N=1000):
    S, I, R = y
    dS = -beta * S * I / N
    dI = beta * S * I / N - gamma * I
    dR = gamma * I
    return [dS, dI, dR]

N = 1000
beta, gamma_sir = 0.3, 0.1
R0 = beta * N / (gamma_sir * N)  # = beta/gamma for SIR
print(f'SIR R0 = {R0:.2f} (epidemic spreads if R0 > 1)')
sir_sol = solve_ivp(sir_model, (0, 160), [999, 1, 0],
                      t_eval=np.linspace(0, 160, 500),
                      args=(beta, gamma_sir, N))

# ---- 3. Repressilator ----
def repressilator(t, y, alpha, alpha0, beta, n):
    # y = [m0, m1, m2, p0, p1, p2]
    m = y[:3]
    p = y[3:]
    dm = np.zeros(3)
    dp = np.zeros(3)
    for i in range(3):
        j = (i - 1) % 3  # gene j represses gene i
        dm[i] = -m[i] + alpha / (1 + p[j]**n) + alpha0
        dp[i] = -beta * (p[i] - m[i])
    return np.concatenate([dm, dp])

rep_params = dict(alpha=216, alpha0=0.216, beta=5.0, n=2)
rep_ic = [0, 0, 0, 2, 1, 0.5]  # slight asymmetry to break symmetry
rep_sol = solve_ivp(repressilator, (0, 100), rep_ic,
                     t_eval=np.linspace(0, 100, 2000),
                     args=tuple(rep_params.values()), max_step=0.1)

if rep_sol.success:
    print(f'Repressilator: max protein conc = {rep_sol.y[3:].max():.2f}')
    print(f'Oscillation detected: {rep_sol.y[3].std() > 1}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel A1: Lotka-Volterra time series
ax = axes[0, 0]
ax.plot(lv_sol.t, lv_sol.y[0], 'steelblue', lw=2, label='Prey (N)')
ax.plot(lv_sol.t, lv_sol.y[1], 'tomato', lw=2, label='Predator (P)')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('A1. Lotka-Volterra — time series')
ax.legend(fontsize=8)

# Panel A2: Lotka-Volterra phase portrait
ax = axes[1, 0]
ax.plot(lv_sol.y[0], lv_sol.y[1], 'k', lw=0.8, alpha=0.7)
ax.plot(lv_sol.y[0, 0], lv_sol.y[1, 0], 'go', ms=8, label='Start')
fp_N = lv_params['gamma'] / lv_params['delta']
fp_P = lv_params['alpha'] / lv_params['beta']
ax.plot(fp_N, fp_P, 'r*', ms=12, label='Fixed point')
ax.set_xlabel('Prey N'); ax.set_ylabel('Predator P')
ax.set_title('A2. Phase portrait (closed orbit = neutral)')
ax.legend(fontsize=8)

# Panel B1: SIR time series
ax = axes[0, 1]
ax.plot(sir_sol.t, sir_sol.y[0], 'steelblue', lw=2, label='S')
ax.plot(sir_sol.t, sir_sol.y[1], 'tomato', lw=2, label='I')
ax.plot(sir_sol.t, sir_sol.y[2], 'green', lw=2, label='R')
ax.axhline(gamma_sir/beta * N, color='grey', ls='--', lw=0.8, label=f'Herd immunity threshold')
ax.set_xlabel('Days'); ax.set_ylabel('People')
ax.set_title(f'B1. SIR Epidemic (R₀={R0:.2f})')
ax.legend(fontsize=7)

# Panel B2: Epidemic peak vs R0
ax = axes[1, 1]
R0_vals = np.linspace(1.05, 5, 30)
peak_I = []
for r0 in R0_vals:
    b = r0 * gamma_sir
    sol_tmp = solve_ivp(sir_model, (0, 300), [999, 1, 0],
                          t_eval=np.linspace(0, 300, 600), args=(b, gamma_sir, N))
    peak_I.append(sol_tmp.y[1].max())
ax.plot(R0_vals, peak_I, 'steelblue', lw=2)
ax.set_xlabel('R₀'); ax.set_ylabel('Peak infected')
ax.set_title('B2. Peak infection vs. R₀')
ax.axvline(1, color='red', ls='--', lw=0.8, label='Epidemic threshold')
ax.legend(fontsize=8)

# Panel C1: Repressilator — protein time series
ax = axes[0, 2]
colors = ['tomato', 'steelblue', 'green']
genes = ['lacI', 'tetR', 'cI']
for i in range(3):
    ax.plot(rep_sol.t, rep_sol.y[3+i], color=colors[i], lw=1.5, label=genes[i])
ax.set_xlabel('Time (au)')
ax.set_ylabel('Protein concentration')
ax.set_title('C1. Repressilator — oscillation')
ax.legend(fontsize=8)

# Panel C2: Repressilator — phase portrait (3D projection)
ax = axes[1, 2]
ax.plot(rep_sol.y[3], rep_sol.y[4], 'k', lw=0.5, alpha=0.6)
ax.set_xlabel('lacI protein'); ax.set_ylabel('tetR protein')
ax.set_title('C2. Repressilator — 2D phase (lacI vs tetR)')
ax.plot(rep_sol.y[3, -1], rep_sol.y[4, -1], 'r*', ms=10, label='End')
ax.legend(fontsize=8)

plt.suptitle('Module 12 NB09: ODE Models of Biological Systems', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ode_models.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Solve for the fixed points of the Lotka-Volterra system analytically. Verify
   numerically that the simulation orbits around the coexistence fixed point.
2. Compute the Jacobian of the repressilator at its symmetric fixed point
   ($m_i = p_i = $ const for all $i$). What are the eigenvalues? When do they
   become complex with positive real part (Hopf bifurcation)?
3. Vary the Hill coefficient $n$ in the repressilator (1, 2, 4). At what minimum
   $n$ do sustained oscillations appear?
4. Modify the SIR model to include waning immunity (some fraction of R returns to S
   with rate $\xi$). What qualitatively new behavior can emerge?

---
## Step 10 — Quiz

1. What is $R_0$ in the SIR model and what does it predict?
2. What is the conserved quantity in the Lotka-Volterra system?
3. What is the repressilator and why is it historically significant?
4. What is a Hopf bifurcation? What does it look like in a phase portrait?
5. How does the Hill coefficient $n$ affect the switch-like behavior of a regulatory function?

---
## Step 12 — Reflection

> *[In one paragraph: explain why mutual repression — as in the repressilator — leads
> to oscillations. What is the key intuition for why three mutually repressing genes
> oscillate but two do not?]*

---
*Next: `10_metabolic_networks_and_fba.ipynb`*